In [1]:
import pandas as pd

In [2]:
# Read Excel
file_path = r"C:\Users\0193284\Downloads\Week 12.xlsx"

df = pd.read_excel(file_path)

# Convert datetime
df["TimeDate"] = pd.to_datetime(df["TimeDate"])

# Sort by Id and Time
df = df.sort_values(["IdNumber", "TimeDate"])

results = []

for id_num, group in df.groupby("IdNumber"):
    arrive_time = None

    for _, row in group.iterrows():

        if row["Direction"] == "Arrive":
            arrive_time = row["TimeDate"]

        elif row["Direction"] == "Depart" and arrive_time is not None:

            minutes = (
                row["TimeDate"] - arrive_time
            ).total_seconds() / 60

            results.append({
                "IdNumber": id_num,
                "ArriveTime": arrive_time,
                "DepartTime": row["TimeDate"],
                "MinutesInTerminal": round(minutes, 2)
            })

            arrive_time = None

result_df = pd.DataFrame(results)

print(result_df)

# Save results
result_df.to_excel("Terminal_Turnaround.xlsx", index=False)

          IdNumber          ArriveTime          DepartTime  MinutesInTerminal
0          08H321U 2026-06-21 15:22:48 2026-06-21 16:59:32              96.73
1    5705165174086 2026-06-17 07:39:26 2026-06-17 08:20:22              40.93
2    5705165174086 2026-06-17 10:59:31 2026-06-17 11:25:38              26.12
3    5705165174086 2026-06-17 13:21:21 2026-06-17 13:50:51              29.50
4    5705165174086 2026-06-17 15:42:46 2026-06-17 16:23:32              40.77
..             ...                 ...                 ...                ...
187  9705035720080 2026-06-18 03:34:06 2026-06-18 04:26:19              52.22
188  9705035720080 2026-06-19 00:38:30 2026-06-19 01:09:27              30.95
189  9705035720080 2026-06-19 03:10:09 2026-06-19 04:12:26              62.28
190  9705035720080 2026-06-20 01:08:57 2026-06-20 01:40:00              31.05
191  9705035720080 2026-06-20 03:44:45 2026-06-20 04:33:31              48.77

[192 rows x 4 columns]


In [3]:
# Total minutes
total_minutes = result_df["MinutesInTerminal"].sum()

# Count of records
count_minutes = result_df["MinutesInTerminal"].count()

# TTT calculation
TTT = total_minutes / count_minutes if count_minutes != 0 else 0

print("Total Minutes In Terminal:", total_minutes)
print("Count:", count_minutes)
print("TTT (Average Turnaround Time):", TTT)

Total Minutes In Terminal: 25798.010000000002
Count: 192
TTT (Average Turnaround Time): 134.3646354166667


In [4]:
fact_terminal = result_df.copy()

fact_terminal["Date"] = fact_terminal["ArriveTime"].dt.date
fact_terminal["Hour"] = fact_terminal["ArriveTime"].dt.hour

# Optional: add shift classification
def get_shift(hour):
    if 6 <= hour < 18:
        return "Day Shift"
    else:
        return "Night Shift"

fact_terminal["Shift"] = fact_terminal["Hour"].apply(get_shift)

# Save for Power BI
fact_terminal.to_excel("Fact_TerminalTurnaround.xlsx", index=False)

In [5]:
summary_terminal = fact_terminal.groupby("IdNumber").agg(
    TotalMinutes=("MinutesInTerminal", "sum"),
    TripCount=("MinutesInTerminal", "count"),
    AvgMinutes=("MinutesInTerminal", "mean")
).reset_index()

summary_terminal["TTT"] = summary_terminal["TotalMinutes"] / summary_terminal["TripCount"]

summary_terminal.to_excel("Summary_TerminalTurnaround.xlsx", index=False)